Como o dataset CMU é separado em arquivos diferentes, incluindo o nome, data e a sinopse dos filmes. Esse primeiro script será utilizado para limpar e unicar os dados necessários em um mesmo arquivo.

In [ ]:
import pandas as pd
import re

path_plots = '~/Downloads/MovieSummaries/plot_summaries.txt'
path_meta = '~/Downloads/MovieSummaries/movie.metadata.tsv'

print("Carregando arquivos brutos...")

plots = pd.read_csv(path_plots, sep='\t', names=['wikipedia_id', 'plot'])

meta_cols = ['wikipedia_id', 'freebase_id', 'title', 'release_date', 
             'box_office', 'runtime', 'languages', 'countries', 'genres']
meta = pd.read_csv(path_meta, sep='\t', names=meta_cols)

df = pd.merge(plots, meta[['wikipedia_id', 'title', 'release_date', 'genres']], on='wikipedia_id', how='inner')

print("Processando linha do tempo e limpando textos...")

df = df.dropna(subset=['release_date'])

df['year'] = df['release_date'].astype(str).str[:4]
df = df[df['year'].str.isnumeric()] 
df['year'] = df['year'].astype(int)

df['half_decade'] = (df['year'] // 5) * 5
df['decade'] = (df['year'] // 10) * 10

def clean_text(text):
    text = str(text).lower()    
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['plot_clean'] = df['plot'].apply(clean_text)

output_path = '~/TCC/cmu_dataset_limpo.csv'
df.to_csv(output_path, index=False)

print("-" * 30)
print(f"Sucesso! Dataset unificado e salvo em: {output_path}")
print(f"Total de filmes processados: {len(df)}")
print("-" * 30)

display(df[['title', 'decade', 'half_decade', 'year', 'plot_clean']].head())

Carregando arquivos brutos...
Processando linha do tempo e limpando textos...
------------------------------
Sucesso! Dataset unificado e salvo em: ~/TCC/cmu_dataset_limpo.csv
Total de filmes processados: 39586
------------------------------


,title,decade,half_decade,year,plot_clean
0,Taxi Blues,1990,1990,1990,"shlykov, a hard-working taxi driver and lyosha..."
1,The Hunger Games,2010,2010,2012,the nation of panem consists of a wealthy capi...
2,Narasimham,2000,2000,2000,poovalli induchoodan is sentenced for six year...
3,The Lemon Drop Kid,1950,1950,1951,"the lemon drop kid , a new york city swindler,..."
4,A Cry in the Dark,1980,1985,1988,seventh-day adventist church pastor michael ch...


Agora irá rodar o modelo BGE-M3 criando os embeddings vindos do dataset limpo.

In [1]:
!pip install safetensors

import torch
print("Versão do PyTorch rodando agora:", torch.__version__)

Versão do PyTorch rodando agora: 2.5.1+cu121


In [ ]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

print("Carregando o dataset limpo...")
df = pd.read_csv('cmu_dataset_limpo.csv')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Processamento ativado via: {device.upper()}")

print("Carregando o modelo BAAI/bge-m3 para a memória da GPU...")
model = SentenceTransformer('BAAI/bge-m3', device=device, model_kwargs={"use_safetensors": True})

textos_para_processar = df['plot_clean'].astype(str).tolist()

print(f"Extraindo vetores de {len(textos_para_processar)} filmes. Acompanhe a barra verde...")
embeddings = model.encode(textos_para_processar, batch_size=16, show_progress_bar=True)

output_vetores = 'embeddings_bge_m3.npy'
np.save(output_vetores, embeddings)

print("-" * 40)
print(f"Sucesso! Vetores salvos no arquivo: {output_vetores}")
print(f"Dimensão da matriz (Filmes x Dimensões do M3): {embeddings.shape}")
print("-" * 40)

/home/thamires/miniconda3/envs/asti-tcc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Carregando o dataset limpo...
Processamento ativado via: CUDA
Carregando o modelo BAAI/bge-m3 para a memória da GPU...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 562.55it/s]


Extraindo vetores de 39586 filmes. Acompanhe a barra verde...


Batches:   0%|          | 0/2475 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 396.00 MiB. GPU 0 has a total capacity of 3.63 GiB of which 267.88 MiB is free. Including non-PyTorch memory, this process has 3.36 GiB memory in use. Of the allocated memory 3.28 GiB is allocated by PyTorch, and 20.97 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [3]:
import torch
print("Versão do PyTorch:", torch.__version__)
print("GPU disponível para o PyTorch?", torch.cuda.is_available())

Versão do PyTorch: 2.5.1+cu121
GPU disponível para o PyTorch? False


In [ ]:
!nvidia-smi

!pip uninstall torch torchvision torchaudio -y

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --force-reinstall

import torch
print("GPU finalmente disponível?", torch.cuda.is_available())

/bin/bash: linha 1: nvidia-smi: comando não encontrado
Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.5.1+cu121
Uninstalling torchaudio-2.5.1+cu121:
  Successfully uninstalled torchaudio-2.5.1+cu121
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached torch-2.5.1%2Bcu121-cp310-cp310-linux_x86_64.whl (780.4 MB)
  Using cached torchvision-0.20.1%2Bcu121-cp310-cp310-linux_x86_64.whl (7.3 MB)
  Using cached torchaudio-2.5.1%2Bcu121-cp310-cp310-linux_x86_64.whl (3.4 MB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.4.0-py3-none-an